**Eye-Tracking & Neurocognitive Intelligence | Reading Comprehension & Neurocognitive Modelling | Presented by: Reza Saadatyar (2026)**

**05 | Generalisation, Robustness and Final Evaluation**

![Generalisation, Robustness and Final Evaluation — scientific workflow](../../images/05_generalisation_and_final_evaluation.svg)

**Objective:** Reconcile held-out prediction scores, select on Validation, and report matched Test performance and conditional participant uncertainty.

**Dataset:** Fixed trial features, participant assignments and saved scores from Notebooks 03 and 04.

**Scope:** Predict the recorded correctness of a paragraph comprehension response from information available by the end of ordinary reading. No post-answer features, clinical inference or cognitive diagnosis.



**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
import os
import sys

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
os.chdir(repo_path)
study_path = repo_path / "reading_comprehension_modelling"
if not (study_path / "code" / "src" / "onestop_io.py").is_file():
    raise FileNotFoundError("Update this checkout using git pull --ff-only origin main before running.")

# --------------------------- 3. Study Dependencies ---------------------------
# Existing compatible Colab packages are reused.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(study_path / "requirements.txt")], check=True)


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("reading_comprehension_modelling", sync_mode="off",
                   module_refresh="off")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

# ----------------- 4. Shared and Project Research Utilities ------------------
from core.output_download import download_outputs_from_colab
from onestop_io import validate_splits, validate_trial_index
from reading_evaluation import evaluate, participant_bootstrap
from sklearn.metrics import confusion_matrix

# ------------------------ 5. Output and Source Paths -------------------------
study = root / "reading_comprehension_modelling"
out_dir = study / "output"
out_dir.mkdir(parents=True, exist_ok=True)
raw_dir = root / "data" / "raw" / "onestop" / "ordinary"
processed = root / "data" / "processed" / "onestop"
processed.mkdir(parents=True, exist_ok=True)
plot_style(font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
           label_size=11.5, label_weight="normal", tick_size=10, legend_size=10,
           figure_dpi=100, figure_size=(10, 5))


**02. Reconcile Matched Held-Out Predictions**

Match every candidate to every required trial, protocol and role. No duplicate model/trial rows or missing coverage are accepted. The primary candidate for each protocol is determined solely by Validation Macro F1, with alphabetical tie-breaking. All other Test results are secondary comparisons.

In [ ]:
# ================== 02. Reconcile Matched Held-Out Predictions ==================

identity = ["trial_id","participant_id","trial_index","article_batch",
            "article_key","paragraph_key","question_key","difficulty_level"]
table = pd.read_csv(out_dir/"02_trial_features.csv",dtype={c:str for c in identity})
index = pd.read_csv(out_dir/"01_trial_index.csv",dtype=str)
index["is_correct"] = index.is_correct.astype(int)
validate_trial_index(index,table)
splits = pd.read_csv(out_dir/"01_participant_splits.csv",dtype=str)
validate_splits(table,splits)
scores = pd.concat([pd.read_csv(out_dir/"03_classical_predictions.csv"),
                    pd.read_csv(out_dir/"04_neural_predictions.csv")],ignore_index=True)
data, validation, selected, test, people, strata, calibration = evaluate(table,splits,scores)
display(selected[["protocol","model","macro_f1","n_people","n_trials"]])


**03. Compare Test Performance and Generalisation**

Report Macro F1 as primary, with balanced accuracy, accuracy, AUROC, incorrect-response precision/recall and AUPRC. Protocol A tests new people within represented batches; Protocol B changes participants and articles together. Their Test cohorts are different, so a performance difference is not an isolated effect of novel text.

In [ ]:
# =============== 03. Compare Test Performance and Generalisation ================

display(test.sort_values(["protocol","macro_f1"],ascending=[True,False]))
fig, axes = plt.subplots(1,2,figsize=(14,8),sharey=True)
for ax,(protocol,g) in zip(axes,test.groupby("protocol")):
    g.sort_values("model").plot.barh(x="model",y="macro_f1",ax=ax,legend=False)
    ax.set(title=protocol,xlabel="Test Macro F1",ylabel="")
fig.tight_layout()
save_figure(fig,out_dir/"05_test_comparison.png")
plt.show()


**04. Analyse Participant Variation and Difficulty**

Report held-out participant metrics and difficulty-specific results. Single-class participant subsets have undefined AUROC and balanced accuracy; they remain missing rather than being replaced with zero. Difficulty strata are observational.

In [ ]:
# =============== 04. Analyse Participant Variation and Difficulty ===============

display(people.head())
display(strata)
primary = test.merge(selected[["protocol","model"]],on=["protocol","model"])
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,row in zip(axes,primary.itertuples(index=False)):
    g=data[(data.protocol==row.protocol)&(data.model==row.model)&data.role.eq("test")]
    cm=confusion_matrix(g.is_correct,g.p_correct>=.5,labels=[0,1])
    ax.imshow(cm,cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    ax.set(xticks=[0,1],yticks=[0,1],xlabel="Predicted",ylabel="Observed",
           title=row.protocol+" / "+row.model)
fig.tight_layout()
save_figure(fig,out_dir/"05_selected_confusion.png")
plt.show()


**05. Quantify Paired Participant Uncertainty**

Resample participant confusion counts jointly across candidates (1,000 draws, seed 17). Report conditional percentile intervals for pooled Macro F1 and matched fusion-versus-unimodal differences. Article dependence, retraining and a held-out batch's population variability are not represented.

In [ ]:
# ================= 05. Quantify Paired Participant Uncertainty ==================

intervals, paired = participant_bootstrap(data,draws=1000,seed=17)
display(intervals)
display(paired)


**06. Assess Score Calibration**

Compare mean predicted scores and observed correct-response fractions in ten fixed bins. Class weighting and a fixed threshold are used; scores are not claimed to be calibrated probabilities. Brier score is a diagnostic, not evidence of calibration by itself.

In [ ]:
# ========================= 06. Assess Score Calibration =========================

fig, axes=plt.subplots(1,2,figsize=(10,4),sharey=True)
for ax,row in zip(axes,selected.itertuples(index=False)):
    g=calibration[(calibration.protocol==row.protocol)&(calibration.model==row.model)]
    ax.plot([0,1],[0,1],linestyle="--",color="grey")
    ax.scatter(g.mean_probability,g.observed_fraction,s=20+g.n_trials/g.n_trials.max()*70)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean score",ylabel="Observed correct fraction",
           title=row.protocol)
fig.tight_layout()
save_figure(fig,out_dir/"05_score_calibration.png")
plt.show()


**07. Save Comparative Evidence and Interpret Scope**

The selected models and secondary comparisons address response correctness for ordinary-reading trials in OneStop. They do not measure general cognitive ability, establish clinical validity or isolate causal gaze effects. Report actual results from the saved tables after execution.

In [ ]:
# ============== 07. Save Comparative Evidence and Interpret Scope ===============

for name,frame in {"05_validation_metrics.csv":validation,"05_selected_models.csv":selected,
                   "05_test_metrics.csv":test,"05_participant_metrics.csv":people,
                   "05_difficulty_metrics.csv":strata,"05_calibration_bins.csv":calibration,
                   "05_participant_intervals.csv":intervals,"05_paired_modality_intervals.csv":paired}.items():
    frame.to_csv(out_dir/name,index=False)
download_outputs_from_colab(out_dir,"05")
